# ReefScan AI — Kaggle GPU training (E1 + E2 + E3)

Runs the **coral bleaching** experiment matrix (feature-extraction screen → fine-tune the top-2) plus **fish segmentation**, fully automatically.

**Before Run All:** right sidebar → *Session options* → **Accelerator: GPU T4 x2** and **Internet: On**. Duration ≈ 1.5–2.5 h on a T4. At the end the artifact zip appears in the **Output panel** — download it.

In [ ]:
# --- 0. Clone (or update) the repo + install dependencies ------------------
# Re-runs auto-update the code (git pull) while PRESERVING your trained
# artefacts inside the clone — never re-download, never lose checkpoints.
import os
repo = "/kaggle/working/reefscan"
if not os.path.exists(repo):
    !git clone https://github.com/Jayant1711/reefscan.git {repo}
else:
    !git -C {repo} pull --ff-only
%cd {repo}
!pip install -q -r requirements.txt

if os.path.basename(os.getcwd()) != os.path.basename(repo):
    os.chdir(repo)  # kernel restarts land elsewhere — re-enter the repo

In [ ]:
# --- 1. GPU + test-suite check (fails loudly if anything is broken) --------
import torch, subprocess, sys
assert torch.cuda.is_available(), "NO GPU — enable the accelerator and re-run!"
print("GPU:", torch.cuda.get_device_name(0))
r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"])
assert r.returncode == 0, "TEST SUITE FAILED — fix before training (do not skip!)"


In [ ]:
# --- 2. Datasets (~8 GB to ephemeral /kaggle/tmp) ---------------------------
!python download_data.py --dataset coral --platform kaggle
!python download_data.py --dataset deepfish --platform kaggle

In [ ]:
# --- 3. Data cleaning (report + quarantine flagged files) -------------------
!python clean_data.py --dataset coral --apply --platform kaggle
!python clean_data.py --dataset seg --apply --platform kaggle

In [ ]:
# --- 4. E1 — coral feature-extraction screen (all 4 backbones, ~35 min) ----
!python train.py --task coral --model all --strategy feature_extraction --platform kaggle
import pandas as pd
pd.read_csv("results/coral/model_leaderboard.csv")

In [ ]:
# --- 5. E2 — auto-pick top-2 from the screen and fine-tune them (~50 min) --
import pandas as pd, subprocess, sys
lb = pd.read_csv("results/coral/model_leaderboard.csv")
fe = lb[lb["strategy"] == "feature_extraction"]
assert len(fe) >= 2, f"FE screen has only {len(fe)}/4 backbones with metrics - cannot pick top-2"
if len(fe) < 4:
    print(f"WARNING: FE screen incomplete ({len(fe)}/4) - proceeding with available backbones")
top2 = fe.nlargest(2, "weighted_f1")["model"].tolist()
print("Fine-tuning top-2 from the FE screen:", top2)
ft_failures = []
for model in top2:
    r = subprocess.run([sys.executable, "train.py", "--task", "coral", "--model", model,
                        "--strategy", "finetune", "--platform", "kaggle"])
    if r.returncode != 0:
        ft_failures.append(model)
        print(f"WARNING: fine-tune failed for {model} - continuing")
print("Fine-tune failures:", ft_failures or "none")
# Optional middle ground on the overall winner:
subprocess.run([sys.executable, "train.py", "--task", "coral", "--model", top2[0],
                "--strategy", "partial_finetune", "--platform", "kaggle"], check=True)
!python evaluate.py --task coral --platform kaggle
pd.read_csv("results/coral/model_leaderboard.csv")

In [ ]:
# --- 6. E3 — fish segmentation U-Net (~30 min) + IoU/Dice/AP metrics -------
!python train.py --task seg --platform kaggle
!python evaluate.py --task seg --platform kaggle
import json; print(json.dumps(json.load(open("results/seg/metrics.json")), indent=2))

In [ ]:
# --- 7. XAI sanity check on the best coral model (Phase-3 innovation) -------
import glob, shlex
sample = (glob.glob("/kaggle/tmp/reefscan_data/coral/imagefolders/test/*/*.jpg") +
          glob.glob("datasets/coral/imagefolders/test/*/*.jpg"))
if sample:
    !python evaluate.py --xai --xai-task coral --image {shlex.quote(str(sample[0]))}
else:
    print("No coral test images found — skipping XAI (not fatal).")

In [ ]:
# --- 9. Save everything: ledger + artifact zip ------------------------------
!python scripts/log_experiment.py
!python scripts/export_artifacts.py
import glob, os, shutil
zip_path = sorted(glob.glob("reefscan_artifacts_kaggle_*.zip"))[-1]
final = os.path.join("/kaggle/working", os.path.basename(zip_path))
if os.path.abspath(zip_path) != os.path.abspath(final):
    shutil.move(zip_path, final)
print("DOWNLOAD NOW (before the session ends):")
print("  Right sidebar -> Output panel ->", os.path.basename(final))
print("  Everything under /kaggle/working persists as notebook output.")
